In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv('/kaggle/input/magnus-carlsen-chess-com-games/magnus_carlsen_games.csv')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------
# PREP
# -------------------------------
df["date"] = pd.to_datetime(df["date"])
df["rating_diff"] = df["player_rating"] - df["opponent_rating"]
df["win_flag"] = (df["result"] == "Win").astype(int)
df["move_count"] = df["moves"].str.count(r"\d+\.")
df["long_game"] = df["move_count"] >= 40
df["first_move"] = df["moves"].str.extract(r"1\. ([a-zA-Z0-9+=#]+)")
df["captures"] = df["moves"].str.count("x")
df["checks"] = df["moves"].str.count("\+")
df["promotions"] = df["moves"].str.count("=")

# -------------------------------
# RESULT DISTRIBUTION
# -------------------------------
df["result"].value_counts().plot(kind="bar", title="Overall Result Distribution")
plt.show()

# -------------------------------
# FORMAT DISTRIBUTION
# -------------------------------
df["format"].value_counts().plot(kind="bar", title="Games by Format")
plt.show()

# -------------------------------
# RESULTS BY FORMAT
# -------------------------------
pd.crosstab(df["format"], df["result"], normalize="index").plot(kind="bar", stacked=True,
                                                               title="Result by Format")
plt.show()

# -------------------------------
# COLOR ADVANTAGE
# -------------------------------
pd.crosstab(df["player_color"], df["result"], normalize="index").plot(
    kind="bar", stacked=True, title="Results by Color")
plt.show()

# -------------------------------
# GAMES PER YEAR
# -------------------------------
df["year"].value_counts().sort_index().plot(
    kind="line", marker="o", title="Games Played per Year")
plt.show()

# -------------------------------
# WIN RATE OVER TIME
# -------------------------------
df.groupby("year")["win_flag"].mean().plot(
    kind="line", marker="o", title="Win Rate Over Time")
plt.show()

# -------------------------------
# PLAYER VS OPPONENT RATINGS
# -------------------------------
df[["player_rating", "opponent_rating"]].plot(
    kind="hist", bins=30, alpha=0.7, title="Rating Distributions")
plt.show()

# -------------------------------
# RATING DIFFERENCE DISTRIBUTION
# -------------------------------
df["rating_diff"].plot(
    kind="hist", bins=40, title="Rating Difference Distribution")
plt.show()

# -------------------------------
# WIN RATE VS RATING DIFFERENCE
# -------------------------------
(df.groupby(pd.cut(df["rating_diff"], [-1000, -200, 0, 200, 1000]))["win_flag"]
 .mean()
 .plot(kind="bar", title="Win Rate vs Rating Difference"))
plt.show()

# -------------------------------
# GAME LENGTH DISTRIBUTION
# -------------------------------
df["move_count"].plot(
    kind="hist", bins=40, title="Game Length Distribution (Moves)")
plt.show()

# -------------------------------
# GAME LENGTH BY RESULT
# -------------------------------
df.groupby("result")["move_count"].mean().plot(
    kind="bar", title="Average Game Length by Result")
plt.show()

# -------------------------------
# LONG VS SHORT GAMES
# -------------------------------
df["long_game"].value_counts(normalize=True).plot(
    kind="bar", title="Long Games vs Short Games")
plt.show()

# -------------------------------
# RESULTS IN LONG GAMES
# -------------------------------
pd.crosstab(df["long_game"], df["result"], normalize="index").plot(
    kind="bar", stacked=True, title="Results in Long vs Short Games")
plt.show()

# -------------------------------
# STRONG OPPONENTS (>=2700)
# -------------------------------
(df[df["opponent_rating"] >= 2700]["result"]
 .value_counts(normalize=True)
 .plot(kind="bar", title="Results vs 2700+ Opponents"))
plt.show()

# -------------------------------
# MOST COMMON FIRST MOVES
# -------------------------------
df["first_move"].value_counts().head(10).plot(
    kind="bar", title="Most Common First Moves")
plt.show()

# -------------------------------
# AGGRESSION METRICS
# -------------------------------
df[["captures", "checks", "promotions"]].mean().plot(
    kind="bar", title="Average Aggression Metrics per Game")
plt.show()

# -------------------------------
# FORMAT-SPECIFIC WIN RATES
# -------------------------------
df.groupby("format")["win_flag"].mean().plot(
    kind="bar", title="Win Rate by Format")
plt.show()

# -------------------------------
# MOST PLAYED OPPONENTS
# -------------------------------
df["opponent_name"].value_counts().head(10).plot(
    kind="bar", title="Most Played Opponents")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# -------------------------------
# PREP (safe re-run)
# -------------------------------
df["hour"] = df["date"].dt.hour
df["month"] = df["date"].dt.month
df["weekday"] = df["date"].dt.day_name()

# -------------------------------
# GAMES BY MONTH (SEASONALITY)
# -------------------------------
df["month"].value_counts().sort_index().plot(
    kind="bar", title="Games by Month")
plt.show()

# -------------------------------
# GAMES BY WEEKDAY
# -------------------------------
df["weekday"].value_counts().plot(
    kind="bar", title="Games by Weekday")
plt.show()

# -------------------------------
# WIN RATE BY MONTH
# -------------------------------
df.groupby("month")["win_flag"].mean().plot(
    kind="line", marker="o", title="Win Rate by Month")
plt.show()

# -------------------------------
# WIN RATE BY WEEKDAY
# -------------------------------
df.groupby("weekday")["win_flag"].mean().plot(
    kind="bar", title="Win Rate by Weekday")
plt.show()

# -------------------------------
# GAME LENGTH VS RATING DIFF
# -------------------------------
plt.scatter(df["rating_diff"], df["move_count"], alpha=0.3)
plt.title("Game Length vs Rating Difference")
plt.xlabel("Rating Difference")
plt.ylabel("Move Count")
plt.show()

# -------------------------------
# WIN / LOSS MOVE LENGTH COMPARISON
# -------------------------------
df[df["result"] == "Win"]["move_count"].plot(
    kind="kde", label="Win")
df[df["result"] == "Loss"]["move_count"].plot(
    kind="kde", label="Loss")
plt.title("Move Length Density: Wins vs Losses")
plt.legend()
plt.show()

# -------------------------------
# FORMAT vs GAME LENGTH
# -------------------------------
df.boxplot(column="move_count", by="format")
plt.title("Game Length by Format")
plt.suptitle("")
plt.show()

# -------------------------------
# RATING DIFF vs RESULT
# -------------------------------
df.boxplot(column="rating_diff", by="result")
plt.title("Rating Difference by Result")
plt.suptitle("")
plt.show()

# -------------------------------
# AGGRESSION vs GAME LENGTH
# -------------------------------
plt.scatter(df["move_count"], df["captures"], alpha=0.3)
plt.title("Captures vs Game Length")
plt.xlabel("Move Count")
plt.ylabel("Captures")
plt.show()

# -------------------------------
# CHECKS vs GAME LENGTH
# -------------------------------
plt.scatter(df["move_count"], df["checks"], alpha=0.3)
plt.title("Checks vs Game Length")
plt.xlabel("Move Count")
plt.ylabel("Checks")
plt.show()

# -------------------------------
# PROMOTIONS (RARE BUT TELLING)
# -------------------------------
df["promotions"].value_counts().plot(
    kind="bar", title="Promotion Count Distribution")
plt.show()

# -------------------------------
# FIRST MOVE vs RESULT
# -------------------------------
(pd.crosstab(df["first_move"], df["result"])
 .loc[df["first_move"].value_counts().head(8).index]
 .plot(kind="bar", stacked=True, title="Results by First Move"))
plt.show()

# -------------------------------
# FIRST MOVE FREQUENCY OVER TIME
# -------------------------------
(df.groupby(["year", "first_move"])
 .size()
 .unstack(fill_value=0)
 .iloc[:, :5]
 .plot(title="Top First Moves Over Time"))
plt.show()

# -------------------------------
# OPPONENT RATING VS RESULT
# -------------------------------
df.boxplot(column="opponent_rating", by="result")
plt.title("Opponent Rating by Result")
plt.suptitle("")
plt.show()

# -------------------------------
# ENDGAME SPECIALIST CHECK
# -------------------------------
df[df["move_count"] >= 50]["result"].value_counts(normalize=True).plot(
    kind="bar", title="Results in Very Long Games (50+ moves)")
plt.show()

# -------------------------------
# FORMAT vs AGGRESSION
# -------------------------------
df.groupby("format")[["captures", "checks"]].mean().plot(
    kind="bar", title="Aggression Metrics by Format")
plt.show()

# -------------------------------
# WIN RATE VS GAME LENGTH BUCKETS
# -------------------------------
(df.groupby(pd.cut(df["move_count"], [0,20,30,40,50,100]))["win_flag"]
 .mean()
 .plot(kind="bar", title="Win Rate vs Game Length Buckets"))
plt.show()


<div style="
    background-color:#fff3e0;
    border:2px dashed #ff9800;
    padding:16px;
    border-radius:8px;
    text-align:center;
    font-size:17px;
    font-weight:bold;
    color:#e65100;
">
✨🚀✨ <br>
<b>Found this helpful?</b><br>
👉 <span style="color:#ff5722;">Please UPVOTE 👍</span> to support this work! 👈  
<br>✨🚀✨
</div>
